In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [6]:
df = pd.read_csv('train.csv')[['Age', 'Fare', 'SibSp', 'Parch', 'Survived']]

In [7]:
df.dropna(inplace=True)

In [10]:
df.head(3)

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0


In [11]:
df['family'] = df['SibSp'] + df['Parch']

In [12]:
df.head(8)

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0
6,54.0,51.8625,0,0,0,0
7,2.0,21.0750,3,1,0,4
8,27.0,11.1333,0,2,1,2


In [14]:
df.drop(columns=['SibSp', 'Parch'], inplace=True)

In [15]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [16]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =0.2, random_state=42)

In [18]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [19]:
# WITHOUT BINARIZATION

clf = DecisionTreeClassifier()

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test, y_pred)

0.6363636363636364

In [20]:
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring= 'accuracy'))

np.float64(0.6471439749608765)

In [21]:
# APPLYING BINARIZATION

from sklearn.preprocessing import Binarizer

In [27]:
'''['family']: This is the specific column in your dataset that the rule applies to. Only the column named "family" gets transformed here.

Binarizer(copy=False): This is a tool that turns numbers into 0s and 1s based on a
cutoff (threshold). Anything above the cutoff becomes a 1, and anything below or 
equal to it becomes a 0.'''

trf = ColumnTransformer([
    ('bin',
     Binarizer(copy=False),
     ['family']) # threshold is family  
], remainder= 'passthrough')

In [45]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [46]:
X_train_trf

array([[  1.    ,  31.    ,  20.525 ],
       [  1.    ,  26.    ,  14.4542],
       [  1.    ,  30.    ,  16.1   ],
       ...,
       [  0.    ,  41.    , 134.5   ],
       [  1.    ,  33.    ,  20.525 ],
       [  0.    ,  33.    ,   7.8958]], shape=(571, 3))

In [47]:
pd.DataFrame(X_train_trf)

,0,1,2
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [48]:
pd.DataFrame(X_train_trf, columns=['family', 'Age', 'Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [54]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf, y_train)

y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test, y_pred2)


0.6153846153846154

In [55]:
X

,Age,Fare,family
0,22.0,7.2500,1
1,38.0,71.2833,1
2,26.0,7.9250,0
3,35.0,53.1000,1
4,35.0,8.0500,0
...,...,...,...
885,39.0,29.1250,1
886,27.0,13.0000,0
887,19.0,30.0000,0
889,26.0,30.0000,0


In [58]:
# Transform the whole dataset (X) because cross_val_score splits and tests it automatically
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X_trf, y, cv=10, scoring= 'accuracy'))

np.float64(0.6303794992175273)